# Corrida en la sala Turing (Windows + GPU, VS Code)

1. Abrir la **carpeta del repositorio** en VS Code (Archivo → Abrir carpeta). Debe tener `build/indice/`.
2. Abrir este cuaderno y elegir como kernel cualquier **Python 3.11** (arriba a la derecha, *Select Kernel*).
   Si VS Code pide instalar `ipykernel`, aceptar.
3. Ajustar la celda 1 y ejecutar todo (**Run All**).

El cuaderno crea `.venv` en el repositorio (si no existe), instala torch y llama-cpp con CUDA
**para Windows** (torch `cu126`, llama-cpp `cu125`) y corre `run.py --gpu --indice-existente` con ese
entorno. La segunda vez no reinstala nada. Si la corrida se interrumpe, se vuelve a ejecutar la
celda 5 y sigue donde quedó.

In [ ]:
# 1. Qué correr
SPLIT = "sample"   # "sample" = las 50 de muestra; "test" = las 992 del sábado (data/test_992.jsonl)
INICIO, FIN = 1, 3 # posiciones del archivo, ambas incluidas (1 = la primera)
TAG = "ensayo"     # resultados en runs/TAG/

In [ ]:
# 2. Ubicación del repositorio, GPU y archivos necesarios
import subprocess, sys, shutil, re
from pathlib import Path

RAIZ = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "run.py").is_file()), None)
assert RAIZ, "No se encuentra run.py: abra en VS Code la carpeta del repositorio."
PY = RAIZ / ".venv" / "Scripts" / "python.exe"
SITIO = RAIZ / ".venv" / "Lib" / "site-packages"
print("Repositorio:", RAIZ)

assert shutil.which("nvidia-smi"), "No hay nvidia-smi: la máquina no tiene GPU NVIDIA o le falta el driver."
print(subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version,memory.total",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout)

assert (RAIZ / "build" / "indice" / "index.faiss").is_file(), "Falta build/indice: copie la carpeta build del paquete."
env = RAIZ / ".env"
if env.is_file():
    for linea in env.read_text(encoding="utf-8", errors="replace").splitlines():
        if linea.strip().upper().startswith("DECODER_") and "qwen" in linea.lower():
            raise SystemExit(".env fija Qwen: borre o cambie las líneas DECODER_* (Gemma es el valor por defecto).")
print("Archivos OK")

In [ ]:
# 3. Entorno .venv con CUDA (solo instala si hace falta; la primera vez tarda varios minutos)
def correr(*cmd):
    print(">", " ".join(map(str, cmd)))
    r = subprocess.run([str(c) for c in cmd], cwd=RAIZ)
    if r.returncode != 0:
        raise SystemExit(f"Falló (código {r.returncode}). Copie el mensaje de arriba.")

def copiar_dll_cuda():
    # llama-cpp en Windows necesita cudart y cublas; se toman de las que trae torch.
    origen, destino = SITIO / "torch" / "lib", SITIO / "llama_cpp" / "lib"
    if origen.is_dir() and destino.is_dir():
        for dll in origen.glob("*.dll"):
            if re.match(r"(cudart|cublas|cublasLt)64_", dll.name):
                shutil.copy2(dll, destino / dll.name)

CHEQUEO = "import torch, llama_cpp; print(torch.cuda.is_available(), llama_cpp.llama_supports_gpu_offload())"

def estado_gpu():
    if not PY.is_file():
        return "sin .venv"
    copiar_dll_cuda()
    r = subprocess.run([str(PY), "-c", CHEQUEO], capture_output=True, text=True, cwd=RAIZ)
    return r.stdout.strip()

if not PY.is_file():
    correr(sys.executable, "-m", "venv", RAIZ / ".venv")

if estado_gpu() != "True True":
    correr(PY, "-m", "pip", "install", "--upgrade", "pip")
    correr(PY, "-m", "pip", "install", "-r", RAIZ / "requirements.txt")
    correr(PY, "-m", "pip", "install", "--force-reinstall", "--no-deps", "torch==2.14.0",
           "--index-url", "https://download.pytorch.org/whl/cu126")
    correr(PY, "-m", "pip", "install", "--force-reinstall", "--no-deps", "llama-cpp-python==0.3.35",
           "--only-binary", "llama-cpp-python",
           "--index-url", "https://abetlen.github.io/llama-cpp-python/whl/cu125")
print("Entorno listo")

In [ ]:
# 4. Verificación: debe decir True True
estado = estado_gpu()
print("torch.cuda / llama_cpp GPU:", estado)
if estado != "True True":
    subprocess.run([str(PY), "-c", CHEQUEO], cwd=RAIZ)
    raise SystemExit("La GPU no quedó disponible: copie el mensaje completo.")

In [ ]:
# 5. Corrida (el avance se ve en vivo; si se interrumpe, volver a ejecutar esta celda)
import os
cmd = [str(PY), "-u", "run.py", "--split", SPLIT, "--gpu", "--indice-existente",
       "--rango", str(INICIO), str(FIN), "--tag", TAG]
print(">", " ".join(cmd))
proc = subprocess.Popen(cmd, cwd=RAIZ, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, encoding="utf-8", errors="replace",
                        env={**os.environ, "PYTHONIOENCODING": "utf-8"})
for linea in proc.stdout:
    print(linea, end="")
print("\nCódigo de salida:", proc.wait())

In [ ]:
# 6. Resultado
salida = RAIZ / "runs" / TAG / "submissions.jsonl"
if salida.is_file():
    n = sum(1 for l in salida.open(encoding="utf-8") if l.strip())
    print(f"{salida}  →  {n} respuestas (se esperaban {FIN - INICIO + 1})")
else:
    print("No se generó", salida)